# YOLOX Fine-Tuning: Add MIL-STD-129 Military Shipping Labels (MSL)

This notebook fine-tunes the existing 97-class hazmat model to add 10 new Military Shipping Label (MSL) classes per MIL-STD-129R standard.

## Overview

**Base Model:** `confidence_boost_epoch_60.pth` (97 hazmat classes, 160 effective epochs)

**Target:** 107 classes (97 hazmat + 10 MSL)

## New MSL Classes (IDs 97-106)

| ID | Class Name | Description |
|----|------------|-------------|
| 97 | mslMilitaryShippingLabel | Full 4x6" MSL with PDF417 barcode |
| 98 | mslPriorityDesignator1 | Priority 1 (highest) - circle with "1" |
| 99 | mslPriorityDesignator2 | Priority 2 (medium) - circle with "2" |
| 100 | mslPriorityDesignator3 | Priority 3 (routine) - circle with "3" |
| 101 | mslPdf417Barcode | PDF417 2D barcode (MIL-STD-129R) |
| 102 | mslLinearBarcode | Code39 linear barcode (legacy) |
| 103 | mslExteriorContainerLabel | Exterior container ID marking |
| 104 | mslIntermediateContainerLabel | Intermediate container ID marking |
| 105 | mslUnitPackLabel | Unit pack ID marking |
| 106 | mslAddressMarkingBlock | FROM/TO/MARK FOR address block |

## Fine-Tuning Strategy

1. **Extend classification head** from 97 to 107 outputs
2. **Initialize new weights** using Xavier initialization
3. **Lower learning rate** (10x lower than confidence boost)
4. **Optional backbone freezing** to preserve hazmat knowledge
5. **Mixed dataset** with both hazmat and MSL images

## Prerequisites

1. Upload base checkpoint to Google Drive:
   - `/My Drive/HazProML/models/confidence_boost_97class/confidence_boost_epoch_60.pth`

2. Prepare MSL training data in YOLO format:
   - `/My Drive/HazProML/data/msl_dataset/images/train/`
   - `/My Drive/HazProML/data/msl_dataset/labels/train/`
   - `/My Drive/HazProML/data/msl_dataset/images/val/`
   - `/My Drive/HazProML/data/msl_dataset/labels/val/`

3. Set runtime to **GPU T4** or better

## Estimated Time
- Setup: 5-10 minutes
- Training (40 epochs): 2-3 hours on T4

## Cell 1: Mount Google Drive & Verify GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU found! Enable in Runtime > Change runtime type > T4 GPU")

## Cell 2: Configuration

### Key Settings for MSL Fine-Tuning:
- `OLD_NUM_CLASSES = 97` - Existing hazmat model
- `NEW_NUM_CLASSES = 107` - With MSL classes added
- `BASIC_LR = 0.00005 / 64` - Very low LR to preserve features
- `FREEZE_BACKBONE_EPOCHS = 10` - Freeze backbone initially

In [ ]:
import os
import json

# ============================================
# PATHS - UPDATE FOR YOUR SETUP
# ============================================

DRIVE_ROOT = "/content/drive/MyDrive"

# Base checkpoint (97-class hazmat model)
BASE_CHECKPOINT = f"{DRIVE_ROOT}/HazProML/models/confidence_boost_97class/confidence_boost_epoch_60.pth"

# Alternative checkpoint paths to try
ALT_CHECKPOINTS = [
    f"{DRIVE_ROOT}/hazmat_models/confidence_boost/confidence_boost_final.pth",
    f"{DRIVE_ROOT}/HazProML/models/confidence_boost_97class/confidence_boost_final.pth",
]

# MSL dataset (your new annotated MSL images)
MSL_DATASET_DIR = f"{DRIVE_ROOT}/HazProML/data/msl_dataset"

# Existing hazmat dataset (to maintain hazmat accuracy)
HAZMAT_DATASET_DIR = f"{DRIVE_ROOT}/HazProML/data/combined_dataset"

# 107-class mapping (97 hazmat + 10 MSL)
CLASS_MAPPING_107 = f"{DRIVE_ROOT}/HazProML/class_mapping_107class_with_msl.json"

# Output directory
DRIVE_OUTPUT = f"{DRIVE_ROOT}/HazProML/models/msl_107class"

# ============================================
# MODEL CONFIGURATION
# ============================================

OLD_NUM_CLASSES = 97   # Existing model classes
NEW_NUM_CLASSES = 107  # With MSL classes
MSL_CLASS_START = 97   # First MSL class ID

# YOLOX-Tiny architecture (must match base model)
DEPTH = 0.33
WIDTH = 0.375
INPUT_SIZE = (640, 640)

# ============================================
# TRAINING PARAMETERS
# ============================================

MAX_EPOCHS = 40
BATCH_SIZE = 16

# Very low LR to preserve hazmat features
# This is 10x lower than confidence boost training
BASIC_LR = 0.00005 / 64.0

# Freeze backbone for first N epochs (helps preserve hazmat knowledge)
FREEZE_BACKBONE_EPOCHS = 10

# Extended no-aug phase for MSL calibration
NO_AUG_EPOCHS = 15

# Warmup
WARMUP_EPOCHS = 3

# Checkpoint backup interval
SAVE_INTERVAL = 5

# ============================================
# DATASET MIXING
# ============================================

# Ratio of MSL images in each batch
# 0.3 = 30% MSL, 70% hazmat (helps balance learning)
MSL_RATIO = 0.3

# Maximum hazmat images to include (to balance dataset)
MAX_HAZMAT_IMAGES = 1500

# ============================================
# LOCAL WORKING DIRECTORY
# ============================================
LOCAL_DATA_DIR = '/content/data/msl_finetune'

# ============================================
# VERIFY PATHS
# ============================================
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Find valid checkpoint
ckpt_path = None
if os.path.exists(BASE_CHECKPOINT):
    ckpt_path = BASE_CHECKPOINT
else:
    for alt in ALT_CHECKPOINTS:
        if os.path.exists(alt):
            ckpt_path = alt
            break

print("=" * 70)
print("MSL FINE-TUNING CONFIGURATION")
print("=" * 70)
print(f"\nBase checkpoint: {ckpt_path}")
if ckpt_path:
    print(f"  Status: Found ({os.path.getsize(ckpt_path)/1024/1024:.1f} MB)")
else:
    print(f"  Status: NOT FOUND - Please upload checkpoint")

print(f"\nMSL Dataset: {MSL_DATASET_DIR}")
print(f"  Status: {'Found' if os.path.exists(MSL_DATASET_DIR) else 'NOT FOUND - Create this directory'}")

print(f"\nHazmat Dataset: {HAZMAT_DATASET_DIR}")
print(f"  Status: {'Found' if os.path.exists(HAZMAT_DATASET_DIR) else 'NOT FOUND'}")

print(f"\n" + "-" * 70)
print("CLASS EXPANSION:")
print("-" * 70)
print(f"  Old classes: {OLD_NUM_CLASSES} (hazmat only)")
print(f"  New classes: {NEW_NUM_CLASSES} (hazmat + MSL)")
print(f"  MSL classes: {NEW_NUM_CLASSES - OLD_NUM_CLASSES} (IDs {MSL_CLASS_START}-{NEW_NUM_CLASSES-1})")

print(f"\n" + "-" * 70)
print("TRAINING SETTINGS:")
print("-" * 70)
print(f"  Epochs: {MAX_EPOCHS}")
print(f"  Learning rate: {BASIC_LR * 64:.6f}")
print(f"  Freeze backbone: First {FREEZE_BACKBONE_EPOCHS} epochs")
print(f"  No-aug phase: Last {NO_AUG_EPOCHS} epochs")
print(f"  MSL ratio: {MSL_RATIO * 100:.0f}%")
print(f"\nOutput: {DRIVE_OUTPUT}")
print("=" * 70)

## Cell 3: Install YOLOX

In [ ]:
%%capture
!pip install cython pycocotools thop loguru tabulate ninja

import os
if not os.path.exists('/content/YOLOX'):
    !git clone https://github.com/Megvii-BaseDetection/YOLOX.git /content/YOLOX

%cd /content/YOLOX
!pip install -v -e .

In [ ]:
import sys
sys.path.insert(0, '/content/YOLOX')

from yolox.exp import get_exp
from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead
print("YOLOX installed successfully!")

## Cell 4: Create 107-Class Mapping

Creates the class mapping file with all 107 classes (97 hazmat + 10 MSL).

In [ ]:
import json

# MSL class definitions per MIL-STD-129R
MSL_CLASSES = {
    "97": {
        "id": 97,
        "name": "mslMilitaryShippingLabel",
        "category": "military_msl",
        "description": "Full MIL-STD-129 Military Shipping Label (4x6 inch) with PDF417 barcode"
    },
    "98": {
        "id": 98,
        "name": "mslPriorityDesignator1",
        "category": "military_msl",
        "description": "Priority Designator 1 - Highest priority (number 1 in circle)"
    },
    "99": {
        "id": 99,
        "name": "mslPriorityDesignator2",
        "category": "military_msl",
        "description": "Priority Designator 2 - Medium priority (number 2 in circle)"
    },
    "100": {
        "id": 100,
        "name": "mslPriorityDesignator3",
        "category": "military_msl",
        "description": "Priority Designator 3 - Routine priority (number 3 in circle)"
    },
    "101": {
        "id": 101,
        "name": "mslPdf417Barcode",
        "category": "military_msl",
        "description": "PDF417 2D barcode per MIL-STD-129R (replaced linear barcodes)"
    },
    "102": {
        "id": 102,
        "name": "mslLinearBarcode",
        "category": "military_msl",
        "description": "Code39 linear barcode (legacy/optional per MIL-STD-129R)"
    },
    "103": {
        "id": 103,
        "name": "mslExteriorContainerLabel",
        "category": "military_msl",
        "description": "Exterior container identification marking"
    },
    "104": {
        "id": 104,
        "name": "mslIntermediateContainerLabel",
        "category": "military_msl",
        "description": "Intermediate container identification marking"
    },
    "105": {
        "id": 105,
        "name": "mslUnitPackLabel",
        "category": "military_msl",
        "description": "Unit pack (smallest container) identification marking"
    },
    "106": {
        "id": 106,
        "name": "mslAddressMarkingBlock",
        "category": "military_msl",
        "description": "FROM/TO/MARK FOR address block with DODAAC codes"
    }
}

# Try to load existing 97-class mapping
class_mapping_97_paths = [
    f"{DRIVE_ROOT}/HazProML/data/combined_dataset/class_mapping.json",
    f"{DRIVE_ROOT}/class_mapping.json",
    f"{HAZMAT_DATASET_DIR}/class_mapping.json",
]

hazmat_mapping = None
for path in class_mapping_97_paths:
    if os.path.exists(path):
        with open(path, 'r') as f:
            hazmat_mapping = json.load(f)
        print(f"Loaded hazmat mapping from: {path}")
        break

if hazmat_mapping is None:
    print("WARNING: No hazmat class mapping found!")
    print("Creating placeholder mapping for classes 0-96...")
    hazmat_mapping = {}
    for i in range(97):
        hazmat_mapping[str(i)] = {
            "id": i,
            "name": f"hazmat_class_{i}",
            "category": "hazmat"
        }

# Combine hazmat + MSL
class_mapping_107 = {}

# Add hazmat classes (0-96)
for key, value in hazmat_mapping.items():
    class_mapping_107[key] = value

# Add MSL classes (97-106)
for key, value in MSL_CLASSES.items():
    class_mapping_107[key] = value

# Save to local and Drive
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
local_mapping_path = f"{LOCAL_DATA_DIR}/class_mapping.json"
with open(local_mapping_path, 'w') as f:
    json.dump(class_mapping_107, f, indent=2)

drive_mapping_path = f"{DRIVE_OUTPUT}/class_mapping_107class.json"
with open(drive_mapping_path, 'w') as f:
    json.dump(class_mapping_107, f, indent=2)

print(f"\nCreated 107-class mapping:")
print(f"  Hazmat classes: 0-96 ({len([k for k in class_mapping_107 if int(k) < 97])} classes)")
print(f"  MSL classes: 97-106 ({len(MSL_CLASSES)} classes)")
print(f"  Total: {len(class_mapping_107)} classes")
print(f"\nSaved to: {local_mapping_path}")
print(f"Saved to: {drive_mapping_path}")

# Display MSL classes
print(f"\n" + "=" * 60)
print("NEW MSL CLASSES (MIL-STD-129R):")
print("=" * 60)
for class_id in range(97, 107):
    cls = class_mapping_107[str(class_id)]
    print(f"  {class_id}: {cls['name']}")
    print(f"       {cls.get('description', '')}")

## Cell 5: Prepare Combined Dataset

Combines hazmat images with MSL images for fine-tuning.

**Important:** Make sure your MSL images are annotated in YOLO format:
- Each image has a `.txt` file with same name
- Format: `class_id x_center y_center width height` (normalized 0-1)
- MSL class IDs should be 97-106

In [ ]:
import os
import shutil
import random
from pathlib import Path
from tqdm import tqdm

# Clean and create directories
!rm -rf /content/data
os.makedirs(f'{LOCAL_DATA_DIR}/images/train', exist_ok=True)
os.makedirs(f'{LOCAL_DATA_DIR}/images/val', exist_ok=True)
os.makedirs(f'{LOCAL_DATA_DIR}/labels/train', exist_ok=True)
os.makedirs(f'{LOCAL_DATA_DIR}/labels/val', exist_ok=True)

def copy_dataset(src_images, src_labels, dst_images, dst_labels, prefix, max_count=None):
    """Copy images and labels with prefix."""
    if not os.path.exists(src_images):
        print(f"  WARNING: Source not found: {src_images}")
        return 0

    files = [f for f in os.listdir(src_images) if f.endswith(('.jpg', '.jpeg', '.png', '.webp'))]
    if max_count:
        random.shuffle(files)
        files = files[:max_count]

    count = 0
    for f in tqdm(files, desc=f"  {prefix}", leave=False):
        # Copy image
        src_img = os.path.join(src_images, f)
        dst_img = os.path.join(dst_images, f"{prefix}_{f}")
        shutil.copy2(src_img, dst_img)

        # Copy label
        label_name = os.path.splitext(f)[0] + '.txt'
        src_label = os.path.join(src_labels, label_name)
        if os.path.exists(src_label):
            dst_label = os.path.join(dst_labels, f"{prefix}_{label_name}")
            shutil.copy2(src_label, dst_label)
        count += 1

    return count

# ============================================
# COPY MSL IMAGES
# ============================================
print("\n" + "=" * 60)
print("COPYING MSL DATASET")
print("=" * 60)

msl_train_count = copy_dataset(
    f"{MSL_DATASET_DIR}/images/train",
    f"{MSL_DATASET_DIR}/labels/train",
    f"{LOCAL_DATA_DIR}/images/train",
    f"{LOCAL_DATA_DIR}/labels/train",
    "msl"
)
print(f"  MSL training images: {msl_train_count}")

msl_val_count = copy_dataset(
    f"{MSL_DATASET_DIR}/images/val",
    f"{MSL_DATASET_DIR}/labels/val",
    f"{LOCAL_DATA_DIR}/images/val",
    f"{LOCAL_DATA_DIR}/labels/val",
    "msl"
)
print(f"  MSL validation images: {msl_val_count}")

# ============================================
# COPY HAZMAT IMAGES (BALANCED)
# ============================================
print("\n" + "=" * 60)
print("COPYING HAZMAT DATASET (BALANCED)")
print("=" * 60)

# Calculate how many hazmat images to include based on MSL ratio
if msl_train_count > 0:
    target_hazmat = int(msl_train_count * (1 - MSL_RATIO) / MSL_RATIO)
    target_hazmat = min(target_hazmat, MAX_HAZMAT_IMAGES)
else:
    target_hazmat = MAX_HAZMAT_IMAGES
    print("  WARNING: No MSL images found, using max hazmat images")

hazmat_train_count = copy_dataset(
    f"{HAZMAT_DATASET_DIR}/images/train",
    f"{HAZMAT_DATASET_DIR}/labels/train",
    f"{LOCAL_DATA_DIR}/images/train",
    f"{LOCAL_DATA_DIR}/labels/train",
    "hazmat",
    max_count=target_hazmat
)
print(f"  Hazmat training images: {hazmat_train_count}")

hazmat_val_count = copy_dataset(
    f"{HAZMAT_DATASET_DIR}/images/val",
    f"{HAZMAT_DATASET_DIR}/labels/val",
    f"{LOCAL_DATA_DIR}/images/val",
    f"{LOCAL_DATA_DIR}/labels/val",
    "hazmat",
    max_count=200
)
print(f"  Hazmat validation images: {hazmat_val_count}")

# ============================================
# SUMMARY
# ============================================
total_train = len(os.listdir(f"{LOCAL_DATA_DIR}/images/train"))
total_val = len(os.listdir(f"{LOCAL_DATA_DIR}/images/val"))

print("\n" + "=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Training: {total_train} images")
print(f"  - MSL: {msl_train_count} ({msl_train_count/max(total_train,1)*100:.1f}%)")
print(f"  - Hazmat: {hazmat_train_count} ({hazmat_train_count/max(total_train,1)*100:.1f}%)")
print(f"\nValidation: {total_val} images")
print(f"  - MSL: {msl_val_count}")
print(f"  - Hazmat: {hazmat_val_count}")

if msl_train_count == 0:
    print("\n" + "!" * 60)
    print("WARNING: NO MSL IMAGES FOUND!")
    print("Please upload MSL images to:")
    print(f"  {MSL_DATASET_DIR}/images/train/")
    print(f"  {MSL_DATASET_DIR}/labels/train/")
    print("!" * 60)

## Cell 6: Convert to COCO Format

In [ ]:
import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from datetime import datetime

def yolo_to_coco(images_dir, labels_dir, class_mapping, output_path, num_classes):
    """
    Convert YOLO annotations to COCO format.
    """
    categories = [
        {
            "id": int(k),
            "name": v["name"],
            "supercategory": v.get("category", "object")
        }
        for k, v in class_mapping.items()
    ]

    images = []
    annotations = []
    annotation_id = 0
    class_counts = {i: 0 for i in range(num_classes)}

    image_files = list(Path(images_dir).glob("*.jpg")) + \
                  list(Path(images_dir).glob("*.jpeg")) + \
                  list(Path(images_dir).glob("*.png"))

    for img_id, img_path in enumerate(tqdm(sorted(image_files), desc="Converting")):
        try:
            with Image.open(img_path) as img:
                width, height = img.size
        except Exception as e:
            print(f"Error reading {img_path}: {e}")
            continue

        images.append({
            "id": img_id,
            "file_name": img_path.name,
            "width": width,
            "height": height,
            "license": 1,
            "date_captured": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

        label_path = Path(labels_dir) / f"{img_path.stem}.txt"
        if not label_path.exists():
            continue

        with open(label_path, 'r') as f:
            content = f.read().strip()

        if not content:
            continue

        for line in content.split('\n'):
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            class_id = int(parts[0])
            if class_id >= num_classes:
                print(f"Warning: class_id {class_id} >= {num_classes} in {label_path}")
                continue

            x_center = float(parts[1])
            y_center = float(parts[2])
            box_width = float(parts[3])
            box_height = float(parts[4])

            # Convert YOLO to COCO format
            x = (x_center - box_width / 2) * width
            y = (y_center - box_height / 2) * height
            w = box_width * width
            h = box_height * height

            annotations.append({
                "id": annotation_id,
                "image_id": img_id,
                "category_id": class_id,
                "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
                "area": round(w * h, 2),
                "iscrowd": 0,
                "segmentation": []
            })
            annotation_id += 1
            class_counts[class_id] += 1

    coco_format = {
        "info": {
            "description": "Hazmat + MSL Label Detection Dataset",
            "version": "1.0",
            "year": 2025,
            "contributor": "HazProML",
            "date_created": datetime.now().strftime("%Y-%m-%d")
        },
        "licenses": [{"id": 1, "name": "MIT", "url": ""}],
        "categories": categories,
        "images": images,
        "annotations": annotations
    }

    with open(output_path, 'w') as f:
        json.dump(coco_format, f)

    return len(images), len(annotations), class_counts

# Convert training set
print("Converting training set...")
train_imgs, train_anns, train_counts = yolo_to_coco(
    f'{LOCAL_DATA_DIR}/images/train',
    f'{LOCAL_DATA_DIR}/labels/train',
    class_mapping_107,
    f'{LOCAL_DATA_DIR}/train.json',
    NEW_NUM_CLASSES
)
print(f"  Images: {train_imgs}")
print(f"  Annotations: {train_anns}")

# Convert validation set
print("\nConverting validation set...")
val_imgs, val_anns, val_counts = yolo_to_coco(
    f'{LOCAL_DATA_DIR}/images/val',
    f'{LOCAL_DATA_DIR}/labels/val',
    class_mapping_107,
    f'{LOCAL_DATA_DIR}/val.json',
    NEW_NUM_CLASSES
)
print(f"  Images: {val_imgs}")
print(f"  Annotations: {val_anns}")

# Show MSL class distribution
print("\n" + "=" * 60)
print("MSL CLASS DISTRIBUTION (Training):")
print("=" * 60)
msl_total = 0
for class_id in range(97, 107):
    count = train_counts[class_id]
    msl_total += count
    class_name = class_mapping_107[str(class_id)]["name"]
    print(f"  {class_id}: {class_name}: {count}")
print(f"\n  Total MSL annotations: {msl_total}")
print(f"  Total Hazmat annotations: {train_anns - msl_total}")

## Cell 7: Create YOLOX Experiment Config

Creates the experiment config for 107-class model with:
- Lower learning rate for fine-tuning
- Extended no-aug phase
- Proper warmup

In [ ]:
exp_content = f'''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
YOLOX Experiment Config - MSL Fine-Tuning (107 classes)

Fine-tunes 97-class hazmat model to add 10 MSL classes.
"""

import os
import torch
from yolox.exp import Exp as MyExp


class Exp(MyExp):
    def __init__(self):
        super(Exp, self).__init__()

        # ---------------- Model Architecture ----------------
        self.depth = {DEPTH}
        self.width = {WIDTH}
        self.num_classes = {NEW_NUM_CLASSES}  # 97 hazmat + 10 MSL
        self.act = "silu"

        # ---------------- Dataset ----------------
        self.data_dir = "{LOCAL_DATA_DIR}"
        self.train_ann = "train.json"
        self.val_ann = "val.json"
        self.data_num_workers = 4

        # ---------------- Training Schedule ----------------
        self.max_epoch = {MAX_EPOCHS}
        self.warmup_epochs = {WARMUP_EPOCHS}

        # Very low LR for fine-tuning (preserve hazmat features)
        self.basic_lr_per_img = {BASIC_LR}

        self.scheduler = "yoloxwarmcos"
        self.min_lr_ratio = 0.01
        self.weight_decay = 5e-4
        self.momentum = 0.9

        # Extended no-aug phase
        self.no_aug_epochs = {NO_AUG_EPOCHS}

        # ---------------- Input Size ----------------
        self.input_size = {INPUT_SIZE}
        self.test_size = {INPUT_SIZE}
        self.random_size = (14, 26)

        # ---------------- Augmentation ----------------
        # Lighter augmentation for fine-tuning
        self.mosaic_prob = 0.5
        self.mixup_prob = 0.3
        self.enable_mixup = True
        self.mosaic_scale = (0.5, 1.5)
        self.mixup_scale = (0.5, 1.5)
        self.hsv_prob = 1.0
        self.flip_prob = 0.5
        self.degrees = 5.0
        self.translate = 0.05
        self.shear = 1.0

        # ---------------- NMS & Confidence ----------------
        self.nmsthre = 0.65
        self.test_conf = 0.01

        # ---------------- Output ----------------
        self.output_dir = "/content/outputs"
        self.exp_name = "yolox_msl_107class"
        self.eval_interval = 1000  # Disable mid-training eval
        self.save_history_ckpt = True
        self.print_interval = 50

    def get_data_loader(self, batch_size, is_distributed, no_aug=False, cache_img=None):
        from yolox.data import COCODataset, TrainTransform, YoloBatchSampler, DataLoader, InfiniteSampler, MosaicDetection, worker_init_reset_seed
        from yolox.utils import wait_for_the_master

        with wait_for_the_master():
            dataset = COCODataset(
                data_dir=self.data_dir,
                json_file=self.train_ann,
                img_size=self.input_size,
                preproc=TrainTransform(max_labels=50, flip_prob=self.flip_prob, hsv_prob=self.hsv_prob),
                cache=False,
                name="images/train"
            )

        dataset = MosaicDetection(
            dataset,
            mosaic=not no_aug,
            img_size=self.input_size,
            preproc=TrainTransform(max_labels=120, flip_prob=self.flip_prob, hsv_prob=self.hsv_prob),
            degrees=self.degrees,
            translate=self.translate,
            mosaic_scale=self.mosaic_scale,
            mixup_scale=self.mixup_scale,
            shear=self.shear,
            enable_mixup=self.enable_mixup,
            mosaic_prob=self.mosaic_prob,
            mixup_prob=self.mixup_prob
        )

        self.dataset = dataset
        sampler = InfiniteSampler(len(self.dataset), seed=self.seed if self.seed else 0)
        batch_sampler = YoloBatchSampler(
            sampler=sampler,
            batch_size=batch_size,
            drop_last=False,
            mosaic=not no_aug
        )

        return DataLoader(
            self.dataset,
            num_workers=self.data_num_workers,
            pin_memory=True,
            batch_sampler=batch_sampler,
            worker_init_fn=worker_init_reset_seed
        )

    def get_eval_loader(self, *args, **kwargs):
        return None

    def get_evaluator(self, *args, **kwargs):
        return None

    def eval(self, model, evaluator, is_distributed, half=False, return_outputs=False):
        return (0, 0, "Evaluation disabled"), None
'''

config_path = '/content/YOLOX/exps/msl_107class_exp.py'
with open(config_path, 'w') as f:
    f.write(exp_content)

print(f"Created experiment config: {config_path}")
print(f"\nConfiguration:")
print(f"  Model: YOLOX-Tiny ({DEPTH}/{WIDTH})")
print(f"  Classes: {NEW_NUM_CLASSES}")
print(f"  Epochs: {MAX_EPOCHS}")
print(f"  Learning rate: {BASIC_LR * 64:.6f}")
print(f"  No-aug epochs: {NO_AUG_EPOCHS}")

## Cell 8: Expand Model Head & Initialize Weights

This is the critical step: we load the 97-class checkpoint and expand the classification head to 107 classes while preserving the learned weights.

**Strategy:**
1. Load 97-class model weights
2. Create 107-class model architecture
3. Copy all compatible weights
4. Initialize new class weights (97-106) with Xavier initialization
5. Save expanded checkpoint for training

In [ ]:
import torch
import torch.nn as nn
from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead

def create_yolox_model(num_classes, depth, width):
    """Create YOLOX model with specified number of classes."""
    in_channels = [256, 512, 1024]
    backbone = YOLOPAFPN(depth, width, in_channels=in_channels, act='silu')
    head = YOLOXHead(num_classes, width, in_channels=in_channels, act='silu')
    model = YOLOX(backbone, head)
    return model

def expand_model_weights(old_state_dict, old_num_classes, new_num_classes):
    """
    Expand model weights from old_num_classes to new_num_classes.

    For classification heads, we:
    1. Keep weights for classes 0 to old_num_classes-1
    2. Initialize weights for new classes with Xavier
    """
    new_state_dict = {}
    expanded_keys = []

    for key, value in old_state_dict.items():
        # Check if this is a classification head weight
        # YOLOX cls_preds have shape [num_classes, in_channels, 1, 1] for Conv2d
        if 'cls_preds' in key:
            if value.shape[0] == old_num_classes:
                # Expand this weight tensor
                old_shape = value.shape
                new_shape = (new_num_classes,) + old_shape[1:]

                # Create new tensor
                new_value = torch.zeros(new_shape, dtype=value.dtype)

                # Copy old weights
                new_value[:old_num_classes] = value

                # Initialize new class weights
                # Use Xavier for conv weights, zeros for biases
                if len(new_shape) > 1 and 'weight' in key:
                    nn.init.xavier_uniform_(new_value[old_num_classes:])
                elif 'bias' in key:
                    # Initialize bias for new classes
                    # Small negative value helps with initial confidence calibration
                    new_value[old_num_classes:] = -2.0

                new_state_dict[key] = new_value
                expanded_keys.append(key)
                print(f"  Expanded: {key} {old_shape} -> {new_shape}")
            else:
                new_state_dict[key] = value
        else:
            # Copy unchanged
            new_state_dict[key] = value

    return new_state_dict, expanded_keys

# ============================================
# LOAD BASE CHECKPOINT
# ============================================
print("=" * 70)
print("EXPANDING MODEL FROM 97 TO 107 CLASSES")
print("=" * 70)

if not ckpt_path or not os.path.exists(ckpt_path):
    raise FileNotFoundError(f"Base checkpoint not found: {ckpt_path}")

print(f"\nLoading base checkpoint: {ckpt_path}")
checkpoint = torch.load(ckpt_path, map_location='cpu')

# Get model state dict
if 'model' in checkpoint:
    old_state_dict = checkpoint['model']
else:
    old_state_dict = checkpoint

print(f"  Loaded state dict with {len(old_state_dict)} keys")

# ============================================
# VERIFY OLD MODEL CLASSES
# ============================================
# Find a cls_preds layer to verify class count
for key, value in old_state_dict.items():
    if 'cls_preds' in key and 'weight' in key:
        detected_classes = value.shape[0]
        print(f"  Detected classes in checkpoint: {detected_classes}")
        if detected_classes != OLD_NUM_CLASSES:
            print(f"  WARNING: Expected {OLD_NUM_CLASSES}, got {detected_classes}")
            OLD_NUM_CLASSES = detected_classes
        break

# ============================================
# EXPAND WEIGHTS
# ============================================
print(f"\nExpanding from {OLD_NUM_CLASSES} to {NEW_NUM_CLASSES} classes...")
new_state_dict, expanded_keys = expand_model_weights(
    old_state_dict,
    OLD_NUM_CLASSES,
    NEW_NUM_CLASSES
)

print(f"\n  Expanded {len(expanded_keys)} weight tensors")

# ============================================
# VERIFY EXPANSION
# ============================================
print(f"\nVerifying expanded model...")
new_model = create_yolox_model(NEW_NUM_CLASSES, DEPTH, WIDTH)

# Load expanded weights
missing, unexpected = new_model.load_state_dict(new_state_dict, strict=False)
if missing:
    print(f"  Missing keys: {len(missing)}")
    for k in missing[:5]:
        print(f"    - {k}")
if unexpected:
    print(f"  Unexpected keys: {len(unexpected)}")

# Test forward pass
new_model.eval()
with torch.no_grad():
    dummy = torch.randn(1, 3, 640, 640)
    output = new_model(dummy)
    print(f"\n  Forward pass successful!")
    print(f"  Output shape: {output.shape}")
    expected_last_dim = 5 + NEW_NUM_CLASSES  # 4 box + 1 obj + classes
    print(f"  Expected last dim: {expected_last_dim} (5 + {NEW_NUM_CLASSES})")
    print(f"  Actual last dim: {output.shape[-1]}")

# ============================================
# SAVE EXPANDED CHECKPOINT
# ============================================
YOLOX_OUTPUT_DIR = '/content/outputs/yolox_msl_107class'
os.makedirs(YOLOX_OUTPUT_DIR, exist_ok=True)

expanded_ckpt_path = f"{YOLOX_OUTPUT_DIR}/expanded_107class_ckpt.pth"

# Create checkpoint with reset epoch
expanded_checkpoint = {
    'model': new_state_dict,
    'start_epoch': 0,  # Reset epoch counter
    'expanded_from': ckpt_path,
    'old_num_classes': OLD_NUM_CLASSES,
    'new_num_classes': NEW_NUM_CLASSES
}

torch.save(expanded_checkpoint, expanded_ckpt_path)
print(f"\nSaved expanded checkpoint: {expanded_ckpt_path}")
print(f"  Size: {os.path.getsize(expanded_ckpt_path)/1024/1024:.1f} MB")

# Also copy to latest_ckpt.pth for training resume
import shutil
shutil.copy(expanded_ckpt_path, f"{YOLOX_OUTPUT_DIR}/latest_ckpt.pth")
print(f"  Copied to: {YOLOX_OUTPUT_DIR}/latest_ckpt.pth")

print("\n" + "=" * 70)
print("MODEL EXPANSION COMPLETE")
print("=" * 70)

## Cell 9: Start MSL Fine-Tuning

Trains the expanded 107-class model.

**Estimated time:** 2-3 hours on T4 GPU

In [ ]:
import threading
import time
import shutil

# Checkpoint backup thread
backup_running = True

def backup_checkpoints():
    """Background thread to backup checkpoints to Google Drive."""
    last_backup_epoch = -1

    while backup_running:
        time.sleep(60)

        latest_ckpt = f"{YOLOX_OUTPUT_DIR}/latest_ckpt.pth"
        if os.path.exists(latest_ckpt):
            try:
                ckpt = torch.load(latest_ckpt, map_location='cpu')
                current_epoch = ckpt.get('start_epoch', 0)

                if current_epoch > 0 and current_epoch % SAVE_INTERVAL == 0 and current_epoch != last_backup_epoch:
                    backup_path = f"{DRIVE_OUTPUT}/msl_107class_epoch_{current_epoch}.pth"
                    shutil.copy(latest_ckpt, backup_path)
                    print(f"\n*** Saved: msl_107class_epoch_{current_epoch}.pth ***")
                    last_backup_epoch = current_epoch

                    # Also update latest on drive
                    shutil.copy(latest_ckpt, f"{DRIVE_OUTPUT}/msl_107class_latest.pth")
            except:
                pass

# Start backup thread
backup_thread = threading.Thread(target=backup_checkpoints, daemon=True)
backup_thread.start()
print("Checkpoint backup thread started")

# Training summary
print("\n" + "=" * 70)
print("MSL FINE-TUNING (107 CLASSES)")
print("=" * 70)
print(f"\nBase model: 97 hazmat classes")
print(f"New model: 107 classes (+ 10 MSL)")
print(f"\nTraining settings:")
print(f"  Epochs: {MAX_EPOCHS}")
print(f"  Learning rate: {BASIC_LR * 64:.6f}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  No-aug epochs: {NO_AUG_EPOCHS}")
print(f"\nDataset:")
print(f"  Training images: {total_train}")
print(f"  MSL images: {msl_train_count}")
print(f"\nCheckpoints saved to: {DRIVE_OUTPUT}")
print("-" * 70 + "\n")

# Start training
%cd /content/YOLOX

!PYTHONPATH=/content/YOLOX python tools/train.py \
    -f exps/msl_107class_exp.py \
    -d 1 \
    -b {BATCH_SIZE} \
    --fp16 \
    -o \
    --resume

# Stop backup thread
backup_running = False

print("\n" + "=" * 70)
print("MSL FINE-TUNING COMPLETE!")
print("=" * 70)

## Cell 10: Save Final Checkpoints

In [ ]:
import shutil
from datetime import datetime

print("Saving final checkpoints to Google Drive...\n")

timestamp = datetime.now().strftime("%Y%m%d")

# Copy final checkpoint
latest_ckpt = f"{YOLOX_OUTPUT_DIR}/latest_ckpt.pth"
if os.path.exists(latest_ckpt):
    # Save with timestamp
    final_name = f"msl_107class_{MAX_EPOCHS}epoch_{timestamp}.pth"
    shutil.copy(latest_ckpt, f"{DRIVE_OUTPUT}/{final_name}")
    print(f"  {final_name}")

    # Save as "final"
    shutil.copy(latest_ckpt, f"{DRIVE_OUTPUT}/msl_107class_final.pth")
    print(f"  msl_107class_final.pth")

# Copy class mapping
shutil.copy(local_mapping_path, f"{DRIVE_OUTPUT}/class_mapping_107class.json")
print(f"  class_mapping_107class.json")

print(f"\nAll files saved to: {DRIVE_OUTPUT}")
print("\nFiles on Drive:")
for f in sorted(os.listdir(DRIVE_OUTPUT)):
    fpath = os.path.join(DRIVE_OUTPUT, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1024 / 1024
        print(f"  {f} ({size:.1f} MB)")

## Cell 11: Test Inference

Run inference on sample images to verify the model detects both hazmat and MSL labels.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from yolox.utils import postprocess
from yolox.data.data_augment import ValTransform

# Load trained model
final_ckpt = f"{DRIVE_OUTPUT}/msl_107class_final.pth"
if not os.path.exists(final_ckpt):
    final_ckpt = f"{YOLOX_OUTPUT_DIR}/latest_ckpt.pth"

print(f"Loading model from: {final_ckpt}")

model = create_yolox_model(NEW_NUM_CLASSES, DEPTH, WIDTH)
ckpt = torch.load(final_ckpt, map_location='cuda')
model.load_state_dict(ckpt['model'])
model.cuda().eval()

print(f"Model loaded (epoch {ckpt.get('start_epoch', 'unknown')})")

# Get class names
CLASSES = [class_mapping_107[str(i)]["name"] for i in range(NEW_NUM_CLASSES)]

# Inference settings
preproc = ValTransform(legacy=False)
conf_thresh = 0.25
nms_thresh = 0.45

# Colors for visualization
np.random.seed(42)
COLORS = np.random.randint(0, 255, size=(NEW_NUM_CLASSES, 3), dtype=np.uint8)
# Make MSL classes distinct (blue tones)
for i in range(97, 107):
    COLORS[i] = [30, 144, 255]  # Dodger blue for MSL

# Get sample images
val_images = list(Path(f'{LOCAL_DATA_DIR}/images/val').glob('*.jpg'))[:8]
if len(val_images) < 8:
    val_images += list(Path(f'{LOCAL_DATA_DIR}/images/val').glob('*.png'))[:8-len(val_images)]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, img_path in zip(axes, val_images):
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    h, w = img.shape[:2]

    # Preprocess
    tensor, _ = preproc(img, None, (640, 640))
    tensor = torch.from_numpy(tensor).unsqueeze(0).float().cuda()

    # Inference
    with torch.no_grad():
        out = model(tensor)
        out = postprocess(out, NEW_NUM_CLASSES, conf_thresh, nms_thresh)

    # Draw results
    num_hazmat = 0
    num_msl = 0

    if out[0] is not None:
        det = out[0].cpu().numpy()
        scale = min(640/h, 640/w)

        for box, score, cls_id in zip(det[:,:4]/scale, det[:,4]*det[:,5], det[:,6].astype(int)):
            x0, y0, x1, y1 = map(int, box)
            color = tuple(map(int, COLORS[cls_id]))

            # Count by type
            if cls_id >= 97:
                num_msl += 1
                color = (255, 144, 30)  # Orange for MSL
            else:
                num_hazmat += 1

            cv2.rectangle(img, (x0, y0), (x1, y1), color, 2)
            label = f"{CLASSES[cls_id][:12]}:{score:.2f}"
            cv2.putText(img, label, (x0, y0-5), cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)

    # Display
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    title = f"H:{num_hazmat} M:{num_msl}"
    if 'msl' in str(img_path).lower():
        title = f"[MSL] {title}"
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.suptitle('MSL Fine-Tuned Model (107 Classes) - H=Hazmat, M=MSL', fontsize=14)
plt.tight_layout()
plt.savefig(f"{DRIVE_OUTPUT}/inference_results.png", dpi=150)
plt.show()

print(f"\nSaved inference results to: {DRIVE_OUTPUT}/inference_results.png")

## Cell 12: Export to ExecuTorch

Convert the trained model to ExecuTorch format for mobile deployment.

In [ ]:
# Install ExecuTorch
!pip install executorch -q

from torch.export import export
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

class YOLOXExportWrapper(nn.Module):
    """Wrapper for YOLOX model export."""
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.model.eval()
        if hasattr(self.model.head, 'decode_in_inference'):
            self.model.head.decode_in_inference = False

    def forward(self, x):
        return self.model(x)

print("=" * 60)
print("EXPORTING TO EXECUTORCH")
print("=" * 60)

# Load final checkpoint
model = create_yolox_model(NEW_NUM_CLASSES, DEPTH, WIDTH)
ckpt = torch.load(final_ckpt, map_location='cpu')
model.load_state_dict(ckpt['model'])
model.eval()

# Wrap for export
export_model = YOLOXExportWrapper(model)
export_model.eval()

# Test forward pass
dummy_input = torch.randn(1, 3, 640, 640)
with torch.no_grad():
    test_output = export_model(dummy_input)
print(f"Output shape: {test_output.shape}")

# Export
print("\nExporting with torch.export...")
example_input = (torch.randn(1, 3, 640, 640),)
exported_program = export(export_model, example_input)

print("Lowering to edge with XNNPACK...")
try:
    edge_program = to_edge_transform_and_lower(
        exported_program,
        partitioner=[XnnpackPartitioner()]
    )
except Exception as e:
    print(f"XNNPACK failed, using basic edge: {e}")
    from executorch.exir import to_edge
    edge_program = to_edge(exported_program)

print("Converting to ExecuTorch...")
executorch_program = edge_program.to_executorch()

# Save
pte_filename = f"yolox_msl_107class_{MAX_EPOCHS}epoch.pte"
pte_path = f"{DRIVE_OUTPUT}/{pte_filename}"

with open(pte_path, 'wb') as f:
    f.write(executorch_program.buffer)

file_size = os.path.getsize(pte_path) / (1024 * 1024)
print(f"\nSaved: {pte_filename} ({file_size:.2f} MB)")
print(f"Location: {pte_path}")

print("\n" + "=" * 60)
print("EXPORT COMPLETE!")
print("=" * 60)

## Cell 13: Summary & Next Steps

In [ ]:
print("=" * 70)
print("MSL FINE-TUNING COMPLETE!")
print("=" * 70)

print(f"\nOutput directory: {DRIVE_OUTPUT}")
print("\nFiles created:")
if os.path.exists(DRIVE_OUTPUT):
    for f in sorted(os.listdir(DRIVE_OUTPUT)):
        fpath = os.path.join(DRIVE_OUTPUT, f)
        if os.path.isfile(fpath):
            size = os.path.getsize(fpath) / 1024 / 1024
            print(f"  {f} ({size:.1f} MB)")

print(f"\n" + "-" * 70)
print("MODEL SUMMARY:")
print("-" * 70)
print(f"  Architecture: YOLOX-Tiny")
print(f"  Total classes: {NEW_NUM_CLASSES}")
print(f"    - Hazmat: 0-96 (97 classes)")
print(f"    - MSL: 97-106 (10 classes)")
print(f"  Training epochs: {MAX_EPOCHS}")
print(f"  Input size: 640x640")

print(f"\n" + "-" * 70)
print("NEXT STEPS:")
print("-" * 70)
print("""
1. Update your React Native app:

   a) Copy .pte file to assets/models/:
      yolox_msl_107class_40epoch.pte

   b) Update src/ml/types/detection.ts:
      numClasses: 107,  // Was 97

   c) Update src/ml/data/class_mapping.json with 107 classes
      (copy class_mapping_107class.json)

   d) Update executorchService.ts MODEL_ASSET path

2. Rebuild the app:
   npx expo run:ios
   npx expo run:android

3. If MSL detection needs improvement:
   - Add more MSL training images
   - Increase MSL_RATIO in training
   - Train for more epochs
""")
print("=" * 70)